<a href="https://colab.research.google.com/github/piccopod/Datalake-Perbankan/blob/main/ETL_with_Google_BigQuery_%26_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas-gbq

# Import Modul

In [2]:
#data manipulation
import pandas as pd
import numpy as np

#data visualization
import seaborn as sns
import matplotlib.pyplot as plt

#connect ke Google Big Query
import pandas_gbq

#akses api key ke akun google bigquery
from google.oauth2 import service_account


#warning
import warnings
warnings.filterwarnings('ignore')


## Function

In [3]:

def handle_duplicate(df):
    df.drop_duplicates(inplace=True)
    print("Duplicate Data sudah dihandle.")
    return df


def identify_missing_value(df):
  df_missing=round(100*df.isna().sum()/len(df),2).reset_index().rename(columns={'index':'Column Name',0:'Missing Value Percentage'})
  df_missing['Missing Value Count']=list(df.isna().sum())
  return df_missing.sort_values(by='Missing Value Count',ascending=False)

def handle_missing_value(df):
  missing=identify_missing_value(df)
  for i in missing['Column Name'].unique(): # loop nama kolom pada dataset
    if float(missing[missing['Column Name']==i]['Missing Value Percentage'])<5: #kondisi saat missing valuenya di bawah 5%
      df.dropna(subset=[i],inplace=True) # hapus baris yang mengandung missing value
    elif float(missing[missing['Column Name']==i]['Missing Value Percentage'])>=5 and float(missing[missing['Column Name']==i]['Missing Value Percentage'])<40: # 5% - 40%
      if df[i].dtype=='object': #object
        df[i].fillna(df[i].mode()[0],inplace=True) #input with mode
      else: #numeric
        df[i].fillna(df[i].median(),inplace=True) #input with median
      continue
    else: #kondisi saat missing value > 40%
      df.drop([i],axis=1,inplace=True) # drop kolom yang mengandung missing value
      continue
  print("Missing Value sudah dihandle.")
  return df


def irrelevant_data(df):
  #perbaiki nama kolom
  df.rename(columns={
      'status_description':'saving_status_description',
      'account_status':'saving_account_status',
      'open_date':'saving_open_date',
      'open_date_1':'loan_open_date',
      'end_date':'loan_end_date',
      'status_id':'loan_account_status',
      'status_description_1':'loan_status_description'
      },inplace=True)
  #drop redundant column
  df.drop(['customer_id_1','customer_id_2','saving_account_id_1','account_location_id'],axis=1, inplace=True)
  #handle marital status irrelevant data
  df['marital_status']=df['marital_status'].replace({"Belom Kawin":"Belum Kawin","Janda/Duda":"Janda"})
  #handle region irrelevant data
  df['region']=df['region'].replace({"Sumatera":"Sumatera Selatan","Sulawesi":"Sulawesi Utara"})
  print("Irrelevant Data sudah dihandle.")
  return df


def drop_outlier(df):
  for i in df.drop(['customer_id', 'saving_account_id','loan_account_id', 'transaction_id','branch_id','saving_account_status','loan_account_status'],axis=1).select_dtypes(include='number').columns:
    Q1=df[i].quantile(0.25)
    Q3=df[i].quantile(0.75)
    IQR=Q3-Q1
    upperbound=Q3+(1.5*IQR)
    lowerbound=Q1-(1.5*IQR)
    df=df[ (df[i]>=lowerbound) & (df[i]<=upperbound)] #drop outlier
  print("Outlier sudah dihandle.")
  return df

def capping_outlier(df):
  for i in df.drop(['customer_id', 'saving_account_id','loan_account_id', 'transaction_id','branch_id','saving_account_status','loan_account_status'],axis=1).select_dtypes(include='number').columns:
    Q1=df[i].quantile(0.25)
    Q3=df[i].quantile(0.75)
    IQR=Q3-Q1
    upperbound=Q3+(1.5*IQR)
    lowerbound=Q1-(1.5*IQR)
    df[i]=np.where(df[i]<lowerbound, lowerbound, df[i]) #capping data dibawah lowerbound
    df[i]=np.where(df[i]>upperbound, upperbound, df[i]) #capping data diatas upperbound
  print("Outlier sudah dihandle.")
  return df


def transform_data(df):
  df=handle_duplicate(df)
  df=handle_missing(df)
  df=irrelevant_data(df)
  df=drop_outlier(df)
  return df

## Big Query Configuration

In [10]:

# BigQuery configurations
key_path = 'trial-bgq-bc10bfcc79c1.json'

scopes = ["https://www.googleapis.com/auth/bigquery"]
credentials = service_account.Credentials.from_service_account_file(
    key_path,
    scopes=scopes
)

project_id = 'trial-bgq'
dataset_id = 'datalake'  # ganti sesuai dataset kamu



## Extract Data From Datamart

In [11]:
sql_query = f"""
select C.* , S.*, L.*, T.*, B.*
from `{project_id}.{dataset_id}.customer` C
left join `{project_id}.{dataset_id}.savingmaster` S
on C.customer_id = S.customer_id
left join `{project_id}.{dataset_id}.loanmaster` L
on C.customer_id = L.customer_id
left join `{project_id}.{dataset_id}.transaction` T
on S.saving_account_id = T.saving_account_id
left join `{project_id}.{dataset_id}.branch` B
on S.account_location_id = B.branch_id
"""

df = pandas_gbq.read_gbq(
    sql_query,
    project_id=project_id,
    credentials=credentials
)

df

Downloading: 100%|██████████|


,customer_id,first_name,last_name,birth_date,address,gender,marital_status,salary,saving_account_id,saving_account_type,...,status_description_1,transaction_id,saving_account_id_1,transaction_type,balance,transaction_amount,transaction_time,branch_id,branch_name,region
0,123456,Ardi,Jono,2001-03-09,Jakarta,L,Belum Kawin,0.0,543219876,Conventional,...,Active,1598765518,543219876,Credit,1500000.0,30000.0,2021-05-20 13:10:00+00:00,2,Kanca Jakarta Selatan,Jakarta
1,123456,Ardi,Jono,2001-03-09,Jakarta,L,Belum Kawin,0.0,543219876,Conventional,...,Active,1598765537,543219876,Credit,4500000.0,90000.0,2021-05-23 15:30:00+00:00,2,Kanca Jakarta Selatan,Jakarta
2,123456,Ardi,Jono,2001-03-09,Jakarta,L,Belum Kawin,0.0,543219876,Conventional,...,Active,1598765432,543219876,Credit,5000000.0,500000.0,2021-05-09 09:30:00+00:00,2,Kanca Jakarta Selatan,Jakarta
3,123456,Ardi,Jono,2001-03-09,Jakarta,L,Belum Kawin,0.0,543219876,Conventional,...,Active,1598765538,543219876,Debit,1200000.0,24000.0,2021-05-24 09:10:00+00:00,2,Kanca Jakarta Selatan,Jakarta
4,123456,Ardi,Jono,2001-03-09,Jakarta,L,Belum Kawin,0.0,543219876,Conventional,...,Active,1598765517,543219876,Debit,3500000.0,70000.0,2021-05-20 11:15:00+00:00,2,Kanca Jakarta Selatan,Jakarta
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
461,123457,Hani,Pulungan,1987-01-23,Jawa Barat,P,Kawin,25000000.0,543219875,Conventional,...,Paid Off,1598765519,543219875,Credit,5000000.0,100000.0,2021-05-20 14:45:00+00:00,3,Kanca Jakarta Pusat,Jakarta
462,123457,Hani,Pulungan,1987-01-23,Jawa Barat,P,Kawin,25000000.0,543219875,Conventional,...,Paid Off,1598765435,543219875,Credit,7400000.0,1000000.0,2021-05-09 15:10:00+00:00,3,Kanca Jakarta Pusat,Jakarta
463,123457,Hani,Pulungan,1987-01-23,Jawa Barat,P,Kawin,25000000.0,543219875,Conventional,...,Paid Off,1598765520,543219875,Debit,2500000.0,50000.0,2021-05-21 09:30:00+00:00,3,Kanca Jakarta Pusat,Jakarta
464,123457,Hani,Pulungan,1987-01-23,Jawa Barat,P,Kawin,25000000.0,543219875,Conventional,...,Paid Off,1598765539,543219875,Debit,3000000.0,60000.0,2021-05-24 10:25:00+00:00,3,Kanca Jakarta Pusat,Jakarta


## Transform Data with Pandas

- Duplicate
- Missing Value
- Irrelevant Data
- Outlier

In [ ]:
df=transform_data(df)
df

Duplicate Data sudah dihandle.
Missing Value sudah dihandle.
Irrelevant Data sudah dihandle.
Outlier sudah dihandle.


,customer_id,first_name,last_name,birth_date,address,gender,marital_status,salary,saving_account_id,saving_account_type,...,loan_account_status,loan_status_description,transaction_id,transaction_type,balance,transaction_amount,transaction_time,branch_id,branch_name,region
1,123459,Dian,Kusuma,1975-12-17,Jawa Timur,P,Belum Kawin,8000000,543219873,Conventional,...,1,Active,1598765524,Credit,1000000.0,20000.0,2021-05-21 14:50:00+00:00,4,Kanca Palu,Sulawesi Utara
2,123459,Dian,Kusuma,1975-12-17,Jawa Timur,P,Belum Kawin,8000000,543219873,Conventional,...,1,Active,1598765543,Credit,3000000.0,60000.0,2021-05-24 14:50:00+00:00,4,Kanca Palu,Sulawesi Utara
3,123459,Dian,Kusuma,1975-12-17,Jawa Timur,P,Belum Kawin,8000000,543219873,Conventional,...,1,Active,1598765544,Debit,1500000.0,30000.0,2021-05-24 15:30:00+00:00,4,Kanca Palu,Sulawesi Utara
4,123459,Dian,Kusuma,1975-12-17,Jawa Timur,P,Belum Kawin,8000000,543219873,Conventional,...,1,Active,1598765523,Debit,2000000.0,40000.0,2021-05-21 13:20:00+00:00,4,Kanca Palu,Sulawesi Utara
6,123459,Dian,Kusuma,1975-12-17,Jawa Timur,P,Belum Kawin,8000000,543219873,Conventional,...,1,Paid Off,1598765524,Credit,1000000.0,20000.0,2021-05-21 14:50:00+00:00,4,Kanca Palu,Sulawesi Utara
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458,123457,Hani,Pulungan,1987-01-23,Jawa Barat,P,Kawin,25000000,543219875,Conventional,...,1,Paid Off,1598765539,Debit,3000000.0,60000.0,2021-05-24 10:25:00+00:00,3,Kanca Jakarta Pusat,Jakarta
460,123457,Hani,Pulungan,1987-01-23,Jawa Barat,P,Kawin,25000000,543219875,Conventional,...,3,Paid Off,1598765540,Credit,1000000.0,20000.0,2021-05-24 11:45:00+00:00,3,Kanca Jakarta Pusat,Jakarta
461,123457,Hani,Pulungan,1987-01-23,Jawa Barat,P,Kawin,25000000,543219875,Conventional,...,3,Paid Off,1598765519,Credit,5000000.0,100000.0,2021-05-20 14:45:00+00:00,3,Kanca Jakarta Pusat,Jakarta
463,123457,Hani,Pulungan,1987-01-23,Jawa Barat,P,Kawin,25000000,543219875,Conventional,...,3,Paid Off,1598765520,Debit,2500000.0,50000.0,2021-05-21 09:30:00+00:00,3,Kanca Jakarta Pusat,Jakarta


## Load Data to Data Warehouse

In [ ]:
# Upload te clean data to BigQuery

dataset_id='data_warehouse'
table_name='customer_data'
pandas_gbq.to_gbq(
    df,
    f'{dataset_id}.{table_name}',
    project_id=project_id,
    if_exists='replace',
    credentials=credentials
)


100%|██████████| 1/1 [00:00<00:00, 6288.31it/s]


In [ ]:
df.to_csv('customer_data.csv',index=False)

In [ ]:
for col in df.select_dtypes(['datetime64[ns, UTC]', 'datetimetz']).columns:
    df[col] = df[col].dt.tz_localize(None)

df.to_excel('customer_data.xlsx',index=False)